# Lab 05 - Blob indexing with Content Understanding

## Required permissions

- **Your identity:** **Search Service Contributor** + **Search Index Data Contributor** on Search and **Storage Blob Data Contributor** on the source container. Sign in with `az login` in your resource tenant.
- **Search system-assigned managed identity:** **Storage Blob Data Reader** on the container and **Cognitive Services User on the parent Foundry resource** for keyless Content Understanding and embeddings. An embedding-only role is not sufficient for Content Understanding.
- **Resources:** an Azure AI Search service with semantic ranker enabled, a Blob container, and a Foundry resource supporting Content Understanding with an embedding deployment. No extra tenant administrator role is required to run the lab.

**Resource prefix:** set `WORKSHOP_PREFIX` in `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Your index, data source, skillset and indexer use `alice01-cu-*`; uploaded blobs use `alice01/content-understanding/`. This keeps the lab separate from notebook 04. Prefixes prevent naming collisions, not enforce access isolation.

**Goal:** extract a two-page PDF as Markdown, chunk it, generate embeddings and retrieve evidence with page ranges. Allow 45-60 minutes. You can run this lab independently of notebook 04.

<img src="../docs/content-understanding-search-pipeline.png" alt="PDF-to-Search pipeline: Blob upload, indexer, Content Understanding, embeddings and provenance, index projections, chunk index, hybrid semantic retrieval and evidence with page ranges" width="1000">

Notebook 04 uses built-in text extraction and Text Split. Here **Content Understanding replaces both extraction and splitting**; there is no second Text Split skill. We use the stable Search REST API `2026-04-01`, not preview semantic chunking or image descriptions. The service returns Markdown content; layout preservation should be inspected, not assumed perfect.

## Why Content Understanding is useful for AI Search

- **More useful document representation:** Markdown can retain headings and table structure rather than flattening everything into plain text. Inspect the sample receipt table to see how well its relationships survive extraction.
- **Retrieval-ready chunks:** one skill extracts and chunks the document, producing `text_sections` that feed directly into embedding and index-projection steps. This reduces the separate extraction and splitting code you need to maintain.
- **Traceable evidence:** location metadata records the source page range and chunk order. Together with the Blob URL, it helps users locate and verify the evidence behind a retrieved result.
- **Fits the Search enrichment pipeline:** the indexer reads the PDF, runs Content Understanding and embeddings, then projects each chunk into its own searchable document. Keyword, vector, hybrid and semantic-ranked queries can use the same indexed content.

This lab uses character-based chunking, not preview semantic chunking. Better document structure and citations do not guarantee correct extraction or answers: inspect the Markdown, page ranges and retrieved evidence.

**Before you begin:** configure your existing resources in `.env` and `.env.azure`, sign in with Azure CLI, and run the Python kernel from `python-notebooks`. Verify the user and Search-identity permissions above and allow the necessary Blob/Foundry network access. The [deployment guide](../infra/README.md) is one option for provisioning the resources and role assignments, not a prerequisite to using your own resources. The sample PDF is original synthetic content; no extra chat-model deployment, API key, account key or SAS is needed.

## 1. Load configuration

Run the kernel from `python-notebooks`, just like labs 01-04. Settings load from `..\.env`, then `..\.env.azure`; both override existing environment variables, and `.env.azure` wins for duplicates. Keep `WORKSHOP_PREFIX` in your untracked `.env` and configure the tenant, Search, Storage, Foundry account and embedding deployment settings listed in the code below. Rerun this cell after changing settings.

The sample PDF is read from `..\data\search\travel-handbook.pdf`. All objects use separate `-cu-` names and a separate Blob folder, so this lab does not overwrite notebook 04's data or require that notebook to have run.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import quote

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv(r'..\.env', override=True)
load_dotenv(r'..\.env.azure', override=True)
root = Path('..')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_STORAGE_RESOURCE_ID',
            'AZURE_STORAGE_BLOB_ENDPOINT', 'AZURE_STORAGE_CONTAINER_NAME',
            'AZURE_AI_ACCOUNT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME', 'WORKSHOP_PREFIX')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Configure .env and .env.azure for your resources.')
prefix = os.environ['WORKSHOP_PREFIX'].strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
storage_endpoint = os.environ['AZURE_STORAGE_BLOB_ENDPOINT'].rstrip('/')
storage_id = os.environ['AZURE_STORAGE_RESOURCE_ID']
container_name = os.environ['AZURE_STORAGE_CONTAINER_NAME']
foundry_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.services.ai.azure.com"
embedding_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
embedding_dimensions = 1536
semantic_configuration = 'chunk-semantic'
api_version = '2026-04-01'
index_name = f'{prefix}-cu-chunks'
data_source_name = f'{prefix}-cu-source'
skillset_name = f'{prefix}-cu-skills'
indexer_name = f'{prefix}-cu-indexer'
blob_prefix = f'{prefix}/content-understanding/'
credential = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
storage_token = get_bearer_token_provider(credential, 'https://storage.azure.com/.default')
print('Search:', search_endpoint)
print('Foundry enrichment:', foundry_endpoint)
print('Index:', index_name, '| source folder:', f'{container_name}/{blob_prefix}')

## 2. Upload the original PDF

Open [`travel-handbook.pdf`](../data/search/travel-handbook.pdf) first. Page 1 includes a receipt-requirements table; page 2 includes exceptions and an example. This is a fictional policy, not advice or a real employer policy. The upload uses `application/pdf`, Entra authentication and your own folder. Repeating it overwrites only this sample blob.

In [ ]:
sample_files = [root / 'data' / 'search' / 'travel-handbook.pdf']
for path in sample_files:
    data = path.read_bytes()
    if not data.startswith(b'%PDF-'):
        raise ValueError(f'Expected a PDF file: {path.name}')
    blob_name = f'{blob_prefix}{path.name}'
    response = requests.put(
        f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
        headers={
            'Authorization': f'Bearer {storage_token()}',
            'x-ms-version': '2023-11-03',
            'x-ms-date': format_datetime(datetime.now(timezone.utc), usegmt=True),
            'x-ms-blob-type': 'BlockBlob', 'Content-Type': 'application/pdf',
        }, data=data, timeout=(10, 90),
    )
    if response.status_code >= 400:
        print(response.text)
    response.raise_for_status()
    print('Uploaded:', blob_name)

## 3. Create the Blob data source

The `ResourceId` connection uses Search's system-assigned managed identity, not your notebook token. The virtual folder limits which files are selected, but is not a participant security boundary. Do not skip this cell: an indexer cannot reference a data source that does not exist. Source deletion detection is deliberately not configured; deleting a blob alone does not remove previously indexed chunks.

In [ ]:
data_source = {
    'name': data_source_name, 'type': 'azureblob',
    'credentials': {'connectionString': f'ResourceId={storage_id};'},
    'container': {'name': container_name, 'query': blob_prefix},
}
response = requests.put(
    f'{search_endpoint}/datasources/{data_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=data_source, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Data source ready:', data_source_name)

## 4. Define the chunk index

Search projections generate `chunk_id` and `parent_id`; do not map these manually. Each chunk stores Markdown, a 1,536-dimensional embedding, filename/Blob URL and Content Understanding location metadata. `page_from` and `page_to` form a **range**, because a chunk may span pages. `ordinal_position` is ordering metadata, not a page number.

In [ ]:
index = {
    'name': index_name,
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'title', 'type': 'Edm.String', 'searchable': True},
        {'name': 'page_from', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'page_to', 'type': 'Edm.Int32', 'filterable': True},
        {'name': 'ordinal_position', 'type': 'Edm.Int32', 'sortable': True},
        {'name': 'location_source', 'type': 'Edm.String'},
        {'name': 'text_vector', 'type': 'Collection(Edm.Single)',
         'searchable': True, 'retrievable': True, 'dimensions': embedding_dimensions,
         'vectorSearchProfile': 'chunk-vector-profile'},
        {'name': 'source', 'type': 'Edm.ComplexType', 'fields': [
            {'name': 'file_name', 'type': 'Edm.String', 'searchable': True},
            {'name': 'url', 'type': 'Edm.String'},
        ]},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'chunk-hnsw', 'kind': 'hnsw',
                        'hnswParameters': {'metric': 'cosine'}}],
        'profiles': [{'name': 'chunk-vector-profile', 'algorithm': 'chunk-hnsw',
                      'vectorizer': 'chunk-vectorizer'}],
        'vectorizers': [{
            'name': 'chunk-vectorizer', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
                'modelName': embedding_model,
            },
        }],
    },
    'semantic': {
        'defaultConfiguration': semantic_configuration,
        'configurations': [{
            'name': semantic_configuration,
            'prioritizedFields': {
                'titleField': {'fieldName': 'title'},
                'prioritizedContentFields': [{'fieldName': 'chunk'}],
            },
        }],
    },
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=index, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Index ready:', index_name)

## 5. Extract, chunk and embed with managed identities

`file_data` provides the original PDF. Content Understanding produces `/document/text_sections/*` objects whose `content` is Markdown. The embedding skill reads each object's **content**, not the whole object. Shaper preserves the original filename and URL.

`cognitiveServices` attaches the Foundry resource for keyless analysis and billing. `identity=None` selects Search's system identity. The same identity calls embeddings when `apiKey` and `authIdentity` are omitted. Fixed-size chunking uses 1,000 characters and 200-character overlap; preview semantic chunking and image descriptions are intentionally not enabled.

See the [Content Understanding skill parameters](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding#skill-parameters) for chunking options and defaults.

In [ ]:
skillset = {
    'name': skillset_name,
    'description': 'Content Understanding PDF extraction, chunking, vectors and page provenance.',
    'cognitiveServices': {
        '@odata.type': '#Microsoft.Azure.Search.AIServicesByIdentity',
        'subdomainUrl': foundry_endpoint, 'identity': None,
    },
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
            'name': 'understand-pdf', 'context': '/document',
            'extractionOptions': ['locationMetadata'],
            'chunkingProperties': {'unit': 'characters', 'maximumLength': 1000, 'overlapLength': 200},
            'inputs': [{'name': 'file_data', 'source': '/document/file_data'}],
            'outputs': [{'name': 'text_sections', 'targetName': 'text_sections'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed-sections', 'context': '/document/text_sections/*',
            'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
            'modelName': embedding_model, 'dimensions': embedding_dimensions,
            'inputs': [{'name': 'text', 'source': '/document/text_sections/*/content'}],
            'outputs': [{'name': 'embedding', 'targetName': 'text_vector'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Util.ShaperSkill',
            'name': 'shape-provenance', 'context': '/document',
            'inputs': [
                {'name': 'file_name', 'source': '/document/metadata_storage_name'},
                {'name': 'url', 'source': '/document/metadata_storage_path'},
            ],
            'outputs': [{'name': 'output', 'targetName': 'provenance'}],
        },
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/text_sections/*',
            'mappings': [
                {'name': 'chunk', 'source': '/document/text_sections/*/content'},
                {'name': 'text_vector', 'source': '/document/text_sections/*/text_vector'},
                {'name': 'title', 'source': '/document/metadata_storage_name'},
                {'name': 'source', 'source': '/document/provenance'},
                {'name': 'page_from', 'source': '/document/text_sections/*/locationMetadata/pageNumberFrom'},
                {'name': 'page_to', 'source': '/document/text_sections/*/locationMetadata/pageNumberTo'},
                {'name': 'ordinal_position', 'source': '/document/text_sections/*/locationMetadata/ordinalPosition'},
                {'name': 'location_source', 'source': '/document/text_sections/*/locationMetadata/source'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
response = requests.put(
    f'{search_endpoint}/skillsets/{skillset_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=skillset, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Skillset ready:', skillset_name)

## 6. Run the Blob indexer

**`allowSkillsetToReadFileData=True` is essential:** it exposes `/document/file_data` to Content Understanding. `dataToExtract='allMetadata'` leaves content extraction to the skill while retaining Blob metadata. The indexer selects PDFs only. Creation starts a run; updating an existing idle indexer explicitly starts a new one. Zero failed-item tolerance prevents partial indexing from being reported as success.

In [ ]:
response = requests.get(
    f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
indexer_exists = response.status_code != 404
previous_run_start = None
if indexer_exists:
    response.raise_for_status()
    previous_run = response.json().get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {
            'dataToExtract': 'allMetadata', 'allowSkillsetToReadFileData': True,
            'indexedFileNameExtensions': '.pdf',
        },
    },
}
response = requests.put(
    f'{search_endpoint}/indexers/{indexer_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, json=indexer, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
print('Indexer ready:', indexer_name)
if indexer_exists:
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('New run requested for the existing indexer.')

In [ ]:
for attempt in range(90):
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    status = response.json()
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')

## 7. Inspect Markdown, page ranges and provenance

Expect several chunks from the one PDF. Compare the receipt table with page 1 and the missing-receipt rules with page 2. A range such as 1-2 means a chunk spans both pages; do not label it as a single page. Inspect the PDF to verify that extracted text and table structure are accurate. The Blob URL still requires authenticated access.

You can also open your index's **Search explorer** in Azure AI Search to inspect the indexed chunks directly. In this example, the returned document includes table content in `chunk`, the source filename in `title`, `page_from=1` and `page_to=2`, ordering/location metadata, and the stored `text_vector`. Compare these fields with the notebook results and original PDF; the screenshot is illustrative, not a required chunk count or ranking.

<img src="../docs/content-understanding-search-explorer.png" alt="Azure AI Search Search Explorer showing extracted table content, source PDF, page range, location metadata and stored embedding" width="900">

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'count': True, 'top': 100, 'orderby': 'ordinal_position',
          'select': 'chunk_id,parent_id,title,chunk,source,page_from,page_to,ordinal_position'},
    timeout=(10, 90),
)
response.raise_for_status()
chunks = response.json()['value']
if not chunks:
    raise RuntimeError('No chunks were indexed. Inspect the indexer errors and source folder.')
if any(not isinstance(item.get('page_from'), int) or not isinstance(item.get('page_to'), int)
       or not 1 <= item['page_from'] <= item['page_to'] <= 2 for item in chunks):
    raise RuntimeError('Expected valid page ranges from the two-page sample; inspect locationMetadata.')
print('Total chunks:', response.json()['@odata.count'])
print(json.dumps(chunks, indent=2))

## 8. Retrieve evidence with hybrid search and semantic ranking

Search generates the query embedding using the index's Azure OpenAI vectorizer. Hybrid retrieval combines keyword and vector candidates; semantic ranking reranks them and produces extractive captions, not an LLM-written answer. Explain the receipt requirements using retrieved text and cite the filename and page range. Check every claim against the PDF.

In [ ]:
question = 'Which travel expenses always need itemized receipts, and what if a hotel receipt is missing?'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'queryType': 'semantic', 'semanticConfiguration': semantic_configuration,
        'captions': 'extractive', 'select': 'chunk_id,title,chunk,source,page_from,page_to', 'top': 3,
    }, timeout=(10, 90),
)
if response.status_code >= 400:
    print(response.text)
response.raise_for_status()
if response.status_code != 200 or response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
if not response.json().get('value'):
    raise RuntimeError('No evidence returned. Inspect the indexed chunks before continuing.')
print(json.dumps(response.json(), indent=2))

## References and next lab

- [Content Understanding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding)
- [Keyless skillset attachment and billing](https://learn.microsoft.com/azure/search/cognitive-search-attach-cognitive-services)
- [Content Understanding regions](https://learn.microsoft.com/azure/ai-services/content-understanding/language-region-support)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Azure OpenAI embedding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-azure-openai-embedding)

Next: [notebook 06](06-push-chunks-vectors.ipynb) chunks TXT files and generates embeddings in Python, then pushes them to a separate index without a Blob indexer.